# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

In [43]:
import tqdm

### Data Preparation

In [2]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: gchhablani/bert-base-cased-finetuned-qqp
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [5]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [6]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [36]:
device = torch.device('cuda')

In [37]:
model.to(device);

In [87]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [46]:
@torch.no_grad()
def evaluate_accuracy(model, val_loader, device):
    model.eval()
    total, correct = 0, 0
    for batch in tqdm.tqdm(val_loader):
      predicted = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
      )
      predicted_labels = predicted.logits.argmax(dim=-1).cpu()
      true_labels = batch['labels'].cpu()
      correct += (true_labels == predicted_labels).sum()
      total += len(true_labels)
    return correct / total

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [48]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(val_set, batch_size=32, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=2)

accuracy = evaluate_accuracy(model, val_loader, device)

100%|██████████| 1264/1264 [04:30<00:00,  4.67it/s]


In [49]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [178]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from transformers.modeling_outputs import SequenceClassifierOutput

ds = load_dataset("SetFit/qqp")

class DuplicateDataCollator:
    def __init__(self, model_name="microsoft/deberta-v3-base", max_length=256, with_labels=True):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.max_length = max_length
        self.with_labels = with_labels

    def __call__(self, examples):
        text1 = [ex["text1"] for ex in examples]
        text2 = [ex["text2"] for ex in examples]

        tok1 = self.tokenizer(text1, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt")
        tok2 = self.tokenizer(text2, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt")

        batch = {
            "input_ids1": tok1["input_ids"],
            "attention_mask1": tok1["attention_mask"],
            "input_ids2": tok2["input_ids"],
            "attention_mask2": tok2["attention_mask"],
        }

        if self.with_labels and "label" in examples[0]:
            batch["labels"] = torch.tensor([ex["label"] for ex in examples], dtype=torch.float)
        if "idx" in examples[0]:
            batch["idx"] = torch.tensor([ex["idx"] for ex in examples], dtype=torch.long, )

        return batch

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

class DuplicateModel(nn.Module):
    def __init__(self, model_name="microsoft/deberta-v3-base", hidden=64, dropout=0.1, freeze_backbone=True, pos_weight=None):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        dim = self.backbone.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(dim * 4, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

        if pos_weight is None:
            pos_weight = torch.tensor([1.0], dtype=torch.float)
        self.register_buffer("pos_weight", pos_weight)
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)

    def encode(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return mean_pooling(out.last_hidden_state, attention_mask)

    def forward(self, input_ids1=None, attention_mask1=None, input_ids2=None, attention_mask2=None, labels=None, idx=None):
        # self.backbone.eval()
        u = self.encode(input_ids1, attention_mask1)
        v = self.encode(input_ids2, attention_mask2)
        u = torch.nn.functional.normalize(u, p=2, dim=1)
        v = torch.nn.functional.normalize(v, p=2, dim=1)

        feats = torch.cat([u, v, torch.abs(u - v), u * v], dim=1)
        feats = feats.float()
        logits = self.classifier(feats).squeeze(-1)

        loss = None
        if labels is not None:
            labels = labels.to(logits.dtype)
            loss = self.loss_fn(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits.unsqueeze(-1))


def compute_metrics(eval_pred):
    logits = eval_pred.predictions.reshape(-1)
    labels = eval_pred.label_ids.reshape(-1)
    probs = 1 / (1 + np.exp(-logits))

    best = {"f1": -1, "thr": 0.5, "acc": 0.0}
    for thr in np.linspace(0.05, 0.95, 181):
        preds = (probs >= thr).astype(np.int32)

        tp = ((preds == 1) & (labels == 1)).sum()
        fp = ((preds == 1) & (labels == 0)).sum()
        fn = ((preds == 0) & (labels == 1)).sum()
        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)

        acc = (preds == labels).mean()
        if f1 > best["f1"]:
            best = {"f1": float(f1), "thr": float(thr), "acc": float(acc)}

    return {"accuracy": best["acc"], "f1": best["f1"], "thr": best["thr"]}

Repo card metadata block was not found. Setting CardData to empty.


In [176]:
model_name = "microsoft/deberta-v3-base"
model = DuplicateModel(model_name=model_name, hidden=64, dropout=0.1, freeze_backbone=True, pos_weight=torch.tensor([1.7]))
model.to(device)
data_collator = DuplicateDataCollator(model_name=model_name, max_length=256, with_labels=True)

args = TrainingArguments(
    output_dir="deberta_biencoder_qqp_frozen",
    learning_rate=1e-3,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=2,
    weight_decay=0.001,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    remove_unused_columns=False,
    logging_steps=600,
    save_steps=600,
    warmup_steps=300
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [177]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Thr
600,0.832195,0.805664,0.584418,0.585872,0.395000
1200,0.801368,0.795209,0.601682,0.594399,0.395000
1800,0.791048,0.786704,0.605442,0.599156,0.455000
2400,0.785147,0.781345,0.604180,0.601291,0.440000
3000,0.779729,0.777309,0.603710,0.602964,0.405000
3600,0.774288,0.773968,0.609325,0.605864,0.420000
4200,0.768229,0.772571,0.601583,0.606719,0.390000
4800,0.769481,0.770934,0.612689,0.607750,0.420000
5400,0.768556,0.770034,0.606629,0.608892,0.425000


TrainOutput(global_step=5686, training_loss=0.7848866288610801, metrics={'train_runtime': 2301.4056, 'train_samples_per_second': 316.195, 'train_steps_per_second': 2.471, 'total_flos': 0.0, 'train_loss': 0.7848866288610801, 'epoch': 2.0})

In [193]:
import os
from safetensors.torch import load_file as safe_load

del model
del trainer
torch.cuda.empty_cache()

ckpt_dir = "deberta_biencoder_qqp_frozen/checkpoint-5400"

model = DuplicateModel(model_name=model_name, hidden=64, dropout=0.1, freeze_backbone=True, pos_weight=torch.tensor([1.7]),).to(device)

safe_path = os.path.join(ckpt_dir, "model.safetensors")

state = safe_load(safe_path)

missing, unexpected = model.load_state_dict(state, strict=False)
print("missing:", missing)
print("unexpected:", unexpected)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


missing: []
unexpected: []


In [194]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

for p in model.backbone.parameters():
    p.requires_grad = True
for p in model.classifier.parameters():
    p.requires_grad = True

backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
head_params = [p for p in model.classifier.parameters() if p.requires_grad]

optimizer = AdamW(
    [
        {"params": backbone_params, "lr": 2e-5, "weight_decay": 0.01},
        {"params": head_params, "lr": 2e-4, "weight_decay": 0.01},
    ]
)

args2 = TrainingArguments(
    output_dir="deberta_biencoder_qqp_phase2",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    remove_unused_columns=False,
    logging_steps=1500,
    save_steps=1500,
)


trainer2 = Trainer(
    model=model,
    args=args2,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
)

In [195]:
trainer2.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [196]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [201]:
!cp -r /content/deberta_biencoder_qqp_frozen/checkpoint-4800 /content/drive/MyDrive/Colab\ Notebooks

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>